# 第 1 周练习：解释我的堆栈跟踪

## 练习目标（理念）

构建一个小工具：输入 **Python 堆栈跟踪（stacktrace）**，输出清晰、简洁的解释和可能修复。

- 使用 **OpenAI API**（经 OpenRouter 兼容端点）
- 用简单的 **提示工程（prompt engineering）**：system 定输出结构，user 放堆栈
- 提供**非流式**与**流式**两种解释函数，便于对比体验

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `openai.chat.completions.create(...)` |
| system / user messages | `SYSTEM_PROMPT` + `USER_PROMPT_PREFIX` + 堆栈文本 |
| 流式输出 | `stream=True` + `update_display` 边生成边刷新 |
| 结构化回答 | Diagnosis / Root Cause / Fix / Confidence / Next Step |

## 怎么跑

1. `.env` 里准备好 `OPENAI_API_KEY`（本笔记本通过 OpenRouter 的 `base_url` 调用）
2. 把示例 `stacktrace` 换成你自己的报错输出
3. 运行流式单元格；或取消最后一格注释改用非流式版本


In [2]:
# ========== 导入：后面解释堆栈要用到的工具 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display、流式时的 update_display
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：后面会指向 OpenRouter 兼容端点
from openai import OpenAI


In [4]:
# ========== 环境 + 客户端：加载密钥并指向 OpenRouter ==========

# 加载 .env：override=True 用文件值覆盖已有环境变量
load_dotenv(override=True)
# 读取 API Key（键名 OPENAI_API_KEY）
api_key = os.getenv('OPENAI_API_KEY')

# ---------- 密钥自检（打印文案保持原样） ----------
if not api_key:
    print('No API key found. Please add OPENAI_API_KEY to your .env file.')
elif api_key.strip() != api_key:
    print('API key has leading/trailing whitespace. Please remove it.')
else:
    print('API key looks good!')

# 创建客户端：显式传入 api_key，并把 base_url 指到 OpenRouter 的 OpenAI 兼容接口
# URL 保持原样，改了就会打到错误服务
openai = OpenAI(
    api_key=api_key,
    base_url='https://openrouter.ai/api/v1',
)


API key looks good!


In [5]:
# ========== 选型：模型名字集中写在一处 ==========

# 使用的聊天模型 id（保持原样；改了会换模型/计费/能力）
MODEL = 'gpt-4.1-mini'


In [6]:
# ========== 示例堆栈跟踪 A：ZeroDivisionError（可换成你自己的） ==========

# stacktrace 字符串保持英文/原始 traceback 格式：这是发给模型的用户材料
# 练习建议：把下面整段替换成你终端里复制出来的真实报错
stacktrace = '''
Traceback (most recent call last):
  File "app.py", line 18, in <module>
    result = divide(10, 0)
  File "app.py", line 7, in divide
    return a / b
ZeroDivisionError: division by zero
'''


In [7]:
# ========== 示例堆栈跟踪 B：TypeError（备用第二个例子） ==========

# 另一个常见错误：对 int 调用 len() —— 可同样丢进 explain 函数练习
stacktrace_type_error = '''
Traceback (most recent call last):
  File "app.py", line 21, in <module>
    print(len(42))
TypeError: object of type 'int' has no len()
'''


In [8]:
# ========== Prompt 模板：system 定输出结构，user 前缀引导任务 ==========

# SYSTEM_PROMPT 保留英文：规定诊断结构（Diagnosis / Root Cause / Fix 等）
SYSTEM_PROMPT = '''
You are a helpful debugging assistant.
Explain Python stacktraces clearly and concisely.
Return in markdown with the following sections (as bullet points):
- Diagnosis (plain English)
- Root Cause Line (quote the line from the traceback if you can)
- Minimal Fix (1-2 lines of code)
- Confidence (Low/Medium/High)
- Next Step (one concrete action)
Keep it short and actionable.
'''

# USER_PROMPT_PREFIX：拼在堆栈正文前面的引导语（同样保持英文）
USER_PROMPT_PREFIX = '''
Here is a Python stacktrace. Explain what happened and how to fix it:

'''


In [9]:
# ========== 简单的根本原因提取器（尽力而为，规则启发式） ==========

def extract_root_cause_line(trace_text: str) -> str:
    # 去掉空行后得到非空行列表：traceback 最后一行通常是异常类型+信息
    lines = [ln.strip() for ln in trace_text.strip().splitlines() if ln.strip()]
    # 更喜欢最后的异常行（例如 ZeroDivisionError: ...）
    if lines:
        return lines[-1]
    # 空输入时返回空字符串
    return ''


In [10]:
# ========== 组装 messages：把堆栈 + 检测到的根因提示交给模型 ==========

def messages_for(trace_text: str):
    # 先本地启发式抽出「最后一行异常」当作提示（不一定完美，但常有用）
    root_cause = extract_root_cause_line(trace_text)
    hint = ''
    if root_cause:
        # 追加到 user 内容末尾；英文字符串保持原样
        hint = f"\n\nRoot cause line (detected): {root_cause}"
    # 返回标准 Chat messages 列表
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_PREFIX + trace_text + hint},
    ]


In [11]:
# ========== 非流式版本：等整段生成完再返回字符串 ==========

def explain_stacktrace(trace_text: str) -> str:
    # 一次性拿到完整 response（不边生成边显示）
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages_for(trace_text),
        max_tokens=800,
    )
    # 取出助手回复正文
    return response.choices[0].message.content


In [13]:
# ========== 流式版本：token 到达时刷新笔记本显示 ==========

def explain_stacktrace_stream(trace_text: str):
    # 先展示本地检测到的根因行（若有）
    root_cause = extract_root_cause_line(trace_text)
    if root_cause:
        display(Markdown(f'**Detected Root Cause:** `{root_cause}`'))
    # stream=True：持续返回增量；max_tokens 限制最长回答
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages_for(trace_text),
        stream=True,
        max_tokens=800,
    )
    # response 累积已生成文本；display_id 用于原地更新同一块输出
    response = ''
    display_handle = display(Markdown(''), display_id=True)
    for chunk in stream:
        # delta.content 可能为 None，用 or '' 避免拼接报错
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

# 对示例 ZeroDivisionError 堆栈跑一遍流式解释
explain_stacktrace_stream(stacktrace)


**Detected Root Cause:** `ZeroDivisionError: division by zero`

```markdown
- Diagnosis
The program tried to divide a number by zero, which is not allowed in Python and caused a runtime error.

- Root Cause Line
`ZeroDivisionError: division by zero`

- Minimal Fix
Add a check before division to avoid dividing by zero:
```python
def divide(a, b):
    if b == 0:
        return "Error: Cannot divide by zero"
    return a / b
```

- Confidence
High

- Next Step
Modify the divide function to handle zero divisor cases gracefully and test with zero and non-zero values.
```

In [ ]:
# ========== 可选：非流式调用（默认注释掉，需要时取消注释） ==========

# 若想用非流式版本，可取消下一行注释（当前保持注释，逻辑与原笔记本一致）：
# display(Markdown(explain_stacktrace(stacktrace)))
